# Criando um Mapas interativo de regiões e renda do DF

Nesse projeto eu queria criar um dataset interativo e acessorasse as pessoas a ter melhores informações de indiciadores socioeconomícos de suas RAs no DF, farei o projeto o melhor possível dentro das minhas possibildiades, mas ressalto que o principal problema é encontrar informações confiáveis e auditáveis para tal projeto.

As informações que peguei foram disponibilizadas pelo Instituto de Pesquisas Estatísticas do Distrito Federa (IPE-DF), mas realizar uma busca nesse site é uma bomba e ta tudo desorganizado.

In [19]:
# Vamos começar importanto as bibliotecas, em especial geopandas e folium para dados geográficos
import pandas as pd
import geopandas as gpd
import folium

In [20]:
gdf = gpd.read_file("Limite_RA_2019.json") # geoJson com os poligonos de cada RA
dados = pd.read_excel("DataSet.xlsx", skiprows=2) # dados sociais

In [21]:
print(gdf.columns)
print(dados.columns)
print(pd.read_excel("DataSet.xlsx", header=None).head(10))

Index(['fid', 'id', 'ra_num', 'ra', 'num_ra', 'st_area_sh', 'legenda',
       'RA_leg', 'geometry'],
      dtype='object')
Index(['NÚMEROS DAS RAs', 'DESCRIÇÃO', 'Valores Absolutos (RS 1,00)',
       'Valores em salários mínimos ', 'índice de Gini', 'Unnamed: 5',
       'numero_filhos'],
      dtype='object')
                                                   0             1  \
0   Renda domiciliar média segundo as Regiões Adm...           NaN   
1                            REGIÕES ADMINISTRATIVAS           NaN   
2                                    NÚMEROS DAS RAs     DESCRIÇÃO   
3                                               RA-I  Plano Piloto   
4                                              RA-II          Gama   
5                                             RA-III    Taguatinga   
6                                              RA-IV    Brazlândia   
7                                              RA-IX     Ceilândia   
8                                               RA-V    Sob

In [22]:
# Renomear colunas para algo simples
dados = dados.rename(columns={
    "DESCRIÇÃO": "ra",
    "Valores Absolutos (RS 1,00)": "renda",
    "índice de Gini": "gini"
})

print(dados.head(10))

  NÚMEROS DAS RAs                  ra         renda  \
0            RA-I        Plano Piloto  16815.024597   
1           RA-II                Gama   3622.426392   
2          RA-III          Taguatinga   4427.833037   
3           RA-IV          Brazlândia   2818.001647   
4           RA-IX           Ceilândia    3235.81864   
5            RA-V          Sobradinho    6753.75393   
6           RA-VI          Planaltina   1865.646922   
7          RA-VII             Paranoá   1941.689217   
8         RA-VIII  Núcleo Bandeirante   4759.750571   
9            RA-X               Guará   7420.551532   

  Valores em salários mínimos       gini  Unnamed: 5  numero_filhos  
0                    11.908658  0.470965         NaN            NaN  
1                     2.565458  0.476452         NaN            NaN  
2                     3.135859  0.472634         NaN            NaN  
3                     1.995752  0.387869         NaN            NaN  
4                     2.291656  0.433131    

In [23]:
# Manter o que interessa
dados = dados[["ra","renda","gini","numero_filhos"]]
dados["renda"] = pd.to_numeric(dados["renda"], errors="coerce")
print(dados.dtypes)

ra                object
renda            float64
gini             float64
numero_filhos    float64
dtype: object


In [25]:
dados["renda"] = dados["renda"].fillna(dados["renda"].mean()) # Trata os NaN
#dados = dados.dropna()
print(dados.head(10))

                   ra         renda      gini  numero_filhos
0        Plano Piloto  16815.024597  0.470965            NaN
1                Gama   3622.426392  0.476452            NaN
2          Taguatinga   4427.833037  0.472634            NaN
3          Brazlândia   2818.001647  0.387869            NaN
4           Ceilândia   3235.818640  0.433131            NaN
5          Sobradinho   6753.753930  0.515685            NaN
6          Planaltina   1865.646922  0.432003            NaN
7             Paranoá   1941.689217  0.309170            NaN
8  Núcleo Bandeirante   4759.750571  0.466706            NaN
9               Guará   7420.551532  0.523741            NaN


In [26]:
# Conferindo se o GeoJSON tem a coluna 'ra'
print(gdf["ra"].head())

0      Plano Piloto
1              Gama
2        Brazlândia
3        Sobradinho
4    Candangolândia
Name: ra, dtype: object


In [27]:
# Fazendo o merge com 'ra'
gdf = gdf.merge(dados, on="ra")
print(gdf.columns)

Index(['fid', 'id', 'ra_num', 'ra', 'num_ra', 'st_area_sh', 'legenda',
       'RA_leg', 'geometry', 'renda', 'gini', 'numero_filhos'],
      dtype='object')


In [17]:
# Criando o mapa

mapa = folium.Map(location=[-15.78,-47.93], zoom_start=10)

folium.Choropleth(
    geo_data=gdf,
    data=gdf,
    columns=["ra","renda"],
    key_on="feature.properties.ra",
    fill_color="RdYlBu",
    legend_name="Renda média"
).add_to(mapa)

# Adiciona os dados num tooltip
folium.GeoJson(
    gdf,
    tooltip=folium.GeoJsonTooltip(
        fields=["ra", "renda", "gini"],
        aliases=["Região:", "Renda média:", "Índice de Gini:"],
        localize=True
    )
).add_to(mapa)

In [18]:
mapa.save("mapa_renda_df.html")

AssertionError: The field ra is not available in the data. Choose from: ().